If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec14_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 14: Simulation Practice and the Monty Hall Problem
v.ekc

Today we cement the simulation recipe — and then use it to settle one of the most famously counterintuitive puzzles in probability. 🚪🚪🚪 (Slides: KL Lecture 14 deck.)

**Today's sections:**
1. Advanced `where`, recapped
2. `for` statements, recapped
3. The dice game, end to end
4. Try It — coin tosses
5. The Monty Hall problem

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. Advanced `where`, Recapped

An array of bools works as a filter directly. (This time our table has names, so the filtering is easier to see.)

In [ ]:
names = make_array('John','Jane','Joe','Jerry','Jill','Jeff','Jim','Jamie')
ages = make_array(16, 22, 18, 15, 19, 15, 16, 21)
age = Table().with_columns('Name', names,
                           'Age', ages)

In [ ]:
age

In [ ]:
age.where('Age', are.above_or_equal_to(18))

In [ ]:
voter = ages >= 18
voter

In [ ]:
age.where(voter)

---
## 2. `for` Statements, Recapped

The loop variable takes each value in turn — and remember, `np.append` does nothing unless you **reassign**. (KL deck, slides 5–6.)

In [ ]:
for pet in make_array('cat', 'dog', 'rabbit'):
    print('I love my ' + pet)

In [ ]:
temporary = make_array('cat', 'dog', 'rabbit')

pet = temporary.item(0)
print('I love my ' + pet)

pet = temporary.item(1)
print('I love my ' + pet)

pet = temporary.item(2)
print('I love my ' + pet)

In [ ]:
for i in np.arange(5):
    print(i)

In [ ]:
for i in np.arange(5):
    print('DATA 111 is AWESOME!')

In [ ]:
s = make_array(2, 3)
np.append(s, 4)
s

In [ ]:
s + 3
s

In [ ]:
s = np.append(s, 4)
s

---
## 3. The Dice Game, End to End

The whole recipe in three cells: one round → one simulated round → ten thousand of them.

In [ ]:
def one_round(my_roll, your_roll):
    if my_roll > your_roll:
        return 1
    elif your_roll > my_roll:
        return -1
    else:
        return 0

In [ ]:
def simulate_one_round():
    my_roll = np.random.choice(np.arange(1,7))
    your_roll = np.random.choice(np.arange(1,7))
    return one_round(my_roll,your_roll)

In [ ]:
game_outcomes = make_array()

for i in np.arange(10000):
    game_outcomes = np.append(game_outcomes, simulate_one_round())
    
game_outcomes

In [ ]:
sum(game_outcomes)

In [ ]:
results = Table().with_column('My winnings', game_outcomes)
results

In [ ]:
results.group('My winnings').barh('My winnings')

---
### Try It 1: 100 coin tosses, again 🪙

(Repetition is the point — this pattern should become automatic.)

1. Make the `coin` array.

2. Simulate a coin toss 100 times.

3. Count the heads in your 100 tosses.

In [ ]:
# Fill in the blank — this cell errors until you do! 😅
coin = ...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same as Lecture 13 — if you didn't need the peek this time, it's working.*

```python
# 1.
coin = make_array('heads', 'tails')

# 2.
np.random.choice(coin, 100)

# 3.
sum(np.random.choice(coin, 100) == 'heads')
```

</details>

Checkpoint — run this, then the full pipeline for reference:

In [ ]:
# run me: needed below
coin = make_array('heads', 'tails')

In [ ]:
# Simulate one outcome
def num_heads():
    return sum(np.random.choice(coin, 100) == 'heads')

In [ ]:
# Decide how many times you want to repeat the experiment
repetitions = 10000

In [ ]:
# Simulate that many outcomes
outcomes = make_array()

for i in np.arange(repetitions):
    outcomes = np.append(outcomes, num_heads())

In [ ]:
heads = Table().with_column('Heads', outcomes)
heads.hist(bins = np.arange(29.5, 70.6))

---
## 4. The Monty Hall Problem 🚪🐐🚗

Three doors: one car, two goats. You pick a door; Monty (who knows where the car is) opens a *different* door revealing a goat, and offers you a switch. **Stay or switch?** Marilyn vos Savant said switch and thousands of PhDs wrote in to call her wrong. Let's simulate instead of arguing. (KL deck, slides 10–15.)

In [ ]:
goats = make_array('first goat', 'second goat')

In [ ]:
def other_goat(x):
    if x == 'first goat':
        return 'second goat'
    elif x == 'second goat':
        return 'first goat'

In [ ]:
[other_goat('first goat'), other_goat('second goat')]

In [ ]:
hidden_behind_doors = np.append(goats, 'car')
hidden_behind_doors

In [ ]:
def monty_hall_game():
    """Return 
    [contestant's guess, what Monty reveals, what remains behind the other door]"""
    
    contestant_guess = np.random.choice(hidden_behind_doors)
    
    if contestant_guess == 'first goat':
        return [contestant_guess, 'second goat', 'car']
    
    if contestant_guess == 'second goat':
        return [contestant_guess, 'first goat', 'car']
    
    if contestant_guess == 'car':
        revealed = np.random.choice(goats)
        return [contestant_guess, revealed, other_goat(revealed)]

In [ ]:
monty_hall_game()

In [ ]:
games = Table(['Guess', 'Revealed', 'Remaining'])

for i in np.arange(10000):
    games.append(monty_hall_game())

In [ ]:
games.show(10)

In [ ]:
original_choice = games.group('Guess')
original_choice

In [ ]:
remaining_door = games.group('Remaining')
remaining_door

In [ ]:
joined = original_choice.join('Guess', remaining_door, 'Remaining')
combined = joined.relabeled(0, 'Item').relabeled(1, 'Original Door').relabeled(2, 'Remaining Door')
combined

In [ ]:
combined.barh(0)

> **The verdict:** the car ends up behind the *other* door about 2/3 of the time — switching doubles your chances. Your first pick is wrong 2/3 of the time, and whenever it's wrong, switching wins. Simulation settles arguments. 🏆

---
> **Today's takeaway:** the simulation recipe is now muscle memory — and it just resolved a puzzle that fooled thousands of mathematicians. Homework 6's roulette section is this exact pattern.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### The simulation recipe (memorize me)
```python
def one_outcome():
    return ...

outcomes = make_array()
for i in np.arange(10000):
    outcomes = np.append(outcomes, one_outcome())
```